In [0]:
# -------------------------------
# Read Bronze Delta Tables as Stream
# -------------------------------
customers_stream = spark.readStream.format("delta") \
    .load("/Volumes/ecom/storage/project/bronze/customers/")

products_stream = spark.readStream.format("delta") \
    .load("/Volumes/ecom/storage/project/bronze/products/")

orders_stream = spark.readStream.format("delta") \
    .load("/Volumes/ecom/storage/project/bronze/orders")

In [0]:
from pyspark.sql.functions import col, to_date, current_timestamp

# -------------------------------
# 1️⃣ Clean Customers
# -------------------------------
clean_customers = customers_stream \
    .dropna(subset=["customer_id", "signup_date"]) \
    .na.fill({
        "first_name": "Unknown",
        "last_name": "Unknown",
        "email": "unknown@example.com"
    }) \
    .withColumn("signup_date", to_date(col("signup_date"), "yyyy-MM-dd")) \
    .withColumn("ingestion_date", current_timestamp()) \
    .withColumn("source_file", col("_metadata.file_path")) \
    .dropDuplicates([
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "country",
        "signup_date"
    ])

# -------------------------------
# 2️⃣ Clean Products
# -------------------------------
clean_products = products_stream \
    .dropna(subset=["product_id"]) \
    .na.fill({
        "product_name": "Unknown",
        "category": "Misc",
        "price": 0.0
    }) \
    .filter(col("price") >= 0) \
    .withColumn("ingestion_date", current_timestamp()) \
    .withColumn("source_file", col("_metadata.file_path")) \
    .dropDuplicates([
        "product_id",
        "product_name",
        "category",
        "price"
    ])

# -------------------------------
# 3️⃣ Clean Orders
# -------------------------------
clean_orders = orders_stream \
    .dropna(subset=["order_id", "customer_id", "product_id", "order_date"]) \
    .na.fill({
        "quantity": 1,
        "total_amount": 0.0
    }) \
    .filter((col("quantity") > 0) & (col("total_amount") >= 0)) \
    .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd")) \
    .withColumn("ingestion_date", current_timestamp()) \
    .withColumn("source_file", col("_metadata.file_path")) \
    .dropDuplicates([
        "order_id",
        "customer_id",
        "product_id",
        "order_date",
        "quantity",
        "total_amount"
    ])


In [0]:
# -------------------------------
#  Write to Silver Delta (Streaming)
# -------------------------------

clean_customers.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/ecom/storage/project/checkpoints/customers_silver/") \
    .partitionBy("signup_date") \
    .trigger(availableNow=True)\
    .start("/Volumes/ecom/storage/project/silver/customers/")

clean_products.writeStream \
    .format("delta") \
    .outputMode("append") \
    .trigger(availableNow=True)\
    .option("checkpointLocation", "/Volumes/ecom/storage/project/checkpoints/products_silver/") \
    .start("/Volumes/ecom/storage/project/silver/products/")

clean_orders.writeStream \
    .format("delta") \
    .outputMode("append") \
    .trigger(availableNow=True)\
    .option("checkpointLocation", "/Volumes/ecom/storage/project/checkpoints/orders_silver/") \
    .partitionBy("order_date") \
    .start("/Volumes/ecom/storage/project/silver/orders/")

In [0]:
# viewing data

spark.read.format("delta") \
    .load("/Volumes/ecom/storage/project/silver/customers/") \
    .show()